# Fine tuning

In [ ]:
#@title Librerías necesarias
import json
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
!pip install unsloth
import unsloth
from unsloth import FastModel
import gc
from tqdm import tqdm
import re
import os
from google.colab import drive

/tmp/ipykernel_2026/1477794334.py:7: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

RECORES_DATASET_PATH = os.path.join(BASE_PATH, "dataset-recores/")
QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_100.json")


Mounted at /content/drive


In [ ]:
#@title Carga del dataset ReCoRES y creación de los conjuntos de entrenamiento, validación y test
import pandas as pd

train_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'train.csv'), sep='\t')
val_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'dev.csv'), sep='\t')
test_df = pd.read_csv(os.path.join(RECORES_DATASET_PATH, 'test.csv'), sep='\t')

print("--- Información del Conjunto de Entrenamiento ---")
train_df.info()


--- Información del Conjunto de Entrenamiento ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1047 entries, 0 to 1046
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      1047 non-null   object
 1   question  1047 non-null   object
 2   A         1047 non-null   object
 3   B         1047 non-null   object
 4   C         1047 non-null   object
 5   D         1047 non-null   object
 6   E         1047 non-null   object
 7   answer    1047 non-null   object
 8   reason    1047 non-null   object
dtypes: object(9)
memory usage: 73.7+ KB


In [ ]:

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it-unsloth-bnb-4bit",
    dtype = None,
    max_seq_length = 4096,
    load_in_4bit = True,
    full_finetuning = False,
)

==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    use_gradient_checkpointing = "unsloth",
    bias = "none",
    random_state = 3407,
)

In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

SYSTEM_PROMPT_BRIEF_REASONING = """Eres un profesor experto en resolver de exámenes de comprensión lectora en español.
Tu tarea es leer el texto y responder ÚNICAMENTE con la letra de la opción correcta (A, B, C, D...).
Ten en cuenta que algunas preguntas o sus opciones de respuesta pueden contener imágenes adjuntas.
Debes analizar exhaustivamente el texto y cualquier imagen proporcionada para determinar cuál es la opción correcta. Si las opciones son imágenes, evalúa cuál de ellas representa correctamente lo descrito en el texto.

No escribas explicaciones, ni introducciones, ni repitas la pregunta.
Debes responder EXCLUSIVAMENTE con un objeto JSON válido, sin incluir explicaciones previas ni posteriores.
El formato debe ser ESTRICTAMENTE este:
{
  "razonamiento": "Aquí escribes una breve explicación de la respuesta elegida basándote en el texto.",
  "respuesta": "Aquí escribes SOLAMENTE la letra de la opción correcta (A, B, C, D...)."
}"""

def build_conversation(row):
    user_content = f"Texto: {row['text']}\nPregunta: {row['question']}\nOpciones:\n"
    user_content += f"A) {row['A']}\nB) {row['B']}\nC) {row['C']}\nD) {row['D']}\nE) {row['E']}"

    assistant_dict = {
        "razonamiento": row['reason'],
        "respuesta": row['answer']
    }
    assistant_content = json.dumps(assistant_dict, ensure_ascii=False, indent=2)

    return [
        {"role": "system", "content": SYSTEM_PROMPT_BRIEF_REASONING},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>') for convo in convos]
    return { "text" : texts }

from datasets import Dataset

def prepare_dataset(df_input):
    """Función para automatizar la limpieza y formateo de cualquier split"""
    df_temp = df_input.copy()
    df_temp["conversations"] = df_temp.apply(build_conversation, axis=1)
    hf_ds = Dataset.from_pandas(df_temp[["conversations"]])

    return hf_ds.map(formatting_prompts_func, batched=True)

train_dataset = prepare_dataset(train_df)
val_dataset   = prepare_dataset(val_df)
test_dataset  = prepare_dataset(test_df)

Map:   0%|          | 0/1047 [00:00<?, ? examples/s]

Map:   0%|          | 0/363 [00:00<?, ? examples/s]

Map:   0%|          | 0/386 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",

        per_device_train_batch_size = 8,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 2,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        eval_strategy = "steps",
        eval_steps = 40,
        save_strategy = "steps",
        save_steps = 40,
        load_best_model_at_end = True,

        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_gemma4_final",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1047 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/363 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,047 | Num Epochs = 2 | Total steps = 132
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 73,400,320 of 8,069,556,768 (0.91% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
40,1.997041,1.593274
80,1.381509,1.503477
120,1.334750,1.494972
132,1.493056,1.493163


In [ ]:
lora_path = os.path.join(BASE_PATH, "model_lora_final")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"Adaptadores guardados en: {lora_path}")

Adaptadores guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/model_lora_final


## Probar si funciona

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

RECORES_DATASET_PATH = os.path.join(BASE_PATH, "dataset-recores/")
QUESTIONS_FILE = os.path.join(BASE_PATH, "data/multiple_choice.json")
ANSWERS_FILE = os.path.join(BASE_PATH, "data/subset_100.json")


In [ ]:

from unsloth import FastVisionModel
model_path = os.path.join(BASE_PATH, "model_lora_final")

# Carga automática: Unsloth trae el modelo base multimodal + tus pesos LoRA
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/model_lora_final",
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)

In [ ]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(QUESTIONS_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
    with open(ANSWERS_FILE, 'r', encoding='utf-8') as f:
        ground_truth = json.load(f)
    return data, ground_truth

In [ ]:
def filter_questions(data, ground_truth):
    """Filtra las preguntas del subset y prepara la lista de tareas a procesar."""
    tareas = []
    for exam in data['exams']:
        nivel = exam['level']
        for ex_wrapper in exam['exercises']:
            exercise = ex_wrapper['exercise']

            for q in exercise['questions']:
                q_id = q['questionId']

                if q_id in ground_truth:
                    tareas.append({
                        "id": q_id,
                        "nivel": nivel,
                        "contexto": exercise.get('text', ''),
                        "pregunta": q['text'],
                        "opciones": q['options'], # Pasamos la lista original intacta
                        "real": ground_truth[q_id]
                    })
    return tareas

In [ ]:
from PIL import Image

def prepare_batch(batch, system_prompt, modo_salida):
    """
    Construye los mensajes en formato multimodal para un lote de tareas.
    Devuelve la lista de mensajes y una lista paralela con las imágenes cargadas.
    """
    mensajes_batch = []
    imagenes_batch = []

    for t in batch:
        user_content = []
        imagenes_tarea = [] # Imágenes específicas para esta tarea/pregunta

        # 1. Añadimos el texto base (contexto y pregunta)
        base_text = f"Texto:\n{t['contexto']}\n\nPregunta: {t['pregunta']}\nOpciones:\n"
        user_content.append({"type": "text", "text": base_text})

        # 2. Iteramos sobre la lista original de opciones (que ahora guardamos intacta)
        for opt in t['opciones']:
            letra = opt['optionId']
            texto = opt.get('text', '').strip()
            ruta_img = opt.get('image-path', '')

            if texto:
                # Si es una opción de texto normal
                user_content.append({"type": "text", "text": f"{letra}) {texto}\n"})

            elif ruta_img:
                # Si es una opción visual (imagen)
                user_content.append({"type": "text", "text": f"{letra}) "})

                # Cargamos la imagen
                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)

                # Insertamos el token visual y un salto de línea
                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        # 3. Añadimos la indicación final según el modo de salida
        if modo_salida == "letra":
            user_content.append({"type": "text", "text": "\nRespuesta:"})

        # 4. Guardamos todo el paquete de esta tarea
        mensajes_batch.append([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [ ]:
import torch

def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    # 1. Aplicamos el template del chat sin tokenizar para obtener las cadenas de texto
    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    # 2. Aplanamos la lista de imágenes
    # batch_imagenes es una lista de listas (ej. [[img_A, img_B], [], [img_A]]).
    # El tokenizador espera una lista simple y secuencial de todas las fotos del lote.
    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]
    tokenizer.padding_side = "left"
    # 3. Tokenizamos enviando tanto el texto como las imágenes
    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # 4. Generación
    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    # 5. Decodificación
    input_len = model_inputs.input_ids.shape[1]
    respuestas_brutas = []

    for output in outputs:
        # Extraemos solo los tokens nuevos generados por el modelo
        gen_tokens = output[input_len:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

In [ ]:
def process_response(texto_bruto, modo_salida):
    """Extrae la letra (A-D) y la explicación según el formato esperado."""
    prediccion = "N/A"
    explicacion = ""
    error_formato = False

    if modo_salida == "json":
        explicacion = texto_bruto # Guardamos el texto bruto por si falla
        try:
            json_match = re.search(r'\{.*\}', texto_bruto, re.DOTALL)
            if json_match:
                datos = json.loads(json_match.group(0))
                letra_raw = datos.get("respuesta", "").strip().upper()
                match_letra = re.search(r'[A-D]', letra_raw)
                prediccion = match_letra.group(0) if match_letra else "N/A"
                explicacion = datos.get("razonamiento", "")
            else:
                error_formato = True
        except Exception:
            error_formato = True

    elif modo_salida == "letra":
        texto_bruto = texto_bruto.upper()
        match = re.search(r'[A-D]', texto_bruto)
        prediccion = match.group(0) if match else "N/A"
        if not match:
            error_formato = True
    return prediccion, explicacion, error_formato

In [ ]:
def show_results(stats, output_file):
    """Imprime por pantalla el resumen de la evaluación."""
    accuracy_total = (stats["aciertos"] / stats["total"]) * 100 if stats["total"] > 0 else 0
    print("\n" + "="*50)
    print(f"RESULTADOS : {output_file}")
    print("="*50)
    if stats["errores_formato"] > 0:
        print(f"Errores de formato (JSON/Regex fallido): {stats['errores_formato']} de {stats['total']}")
    print(f"Accuracy Global: {accuracy_total:.2f}% ({stats['aciertos']}/{stats['total']})")
    print("-" * 50)
    for nivel, s in sorted(stats["por_nivel"].items()):
        acc_n = (s["aciertos"] / s["total"]) * 100
        print(f"Nivel {nivel}: {acc_n:.2f}% ({s['aciertos']}/{s['total']})")

In [ ]:
def evaluate_model(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size=4,
    output_file="resultados.jsonl"
):
    """Función principal que orquesta todo el flujo."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data, ground_truth = load_data()
    tareas = filter_questions(data, ground_truth)

    resultados_finales = []
    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    if os.path.exists(output_file):
        os.remove(output_file)

    for i in tqdm(range(0, len(tareas), batch_size), desc="Progreso"):
        batch = tareas[i : i + batch_size]

        mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
        textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

        batch_results = []

        for j, texto_bruto in enumerate(textos_generados):
            prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

            tarea_actual = batch[j]
            real = tarea_actual["real"]
            nivel = tarea_actual["nivel"]
            es_correcto = (prediccion == real)

            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1
            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1


            if errors:
                stats["errores_formato"] += 1

            batch_results.append({
                "questionId": tarea_actual["id"],
                "nivel": nivel,
                "pregunta": tarea_actual["pregunta"],
                "respuesta_real": real,
                "prediccion_modelo": prediccion,
                "explicacion": explicacion,
                "error_procesamiento_json": errors,
                "estado": "CORRECTO" if es_correcto else "INCORRECTO"
            })

        with open(output_file, 'a', encoding='utf-8') as f:
          for resultado in batch_results:
              linea_json = json.dumps(resultado, ensure_ascii=False)
              f.write(linea_json + '\n')

    show_results(stats, output_file)

In [ ]:
def split_tasks_by_modality(tareas):
    """Separa las tareas en dos grupos: de solo texto y con imágenes."""
    tareas_texto = []
    tareas_imagen = []

    for t in tareas:
        tiene_imagen = False
        if isinstance(t.get('opciones'), list):
            tiene_imagen = any(opt.get('image-path', '') != '' for opt in t['opciones'])

        if tiene_imagen:
            tareas_imagen.append(t)
        else:
            tareas_texto.append(t)

    return tareas_texto, tareas_imagen


def run_inference(
    model,
    tokenizer,
    system_prompt,
    modo_salida="json",
    max_new_tokens=1000,
    batch_size_texto=4, # Batch size solo para textos
    output_file="resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 1. Cargar y dividir tareas
    data, ground_truth = load_data()
    todas_las_tareas = filter_questions(data, ground_truth)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todas_las_tareas)

    # Limpiar archivo previo si existe
    if os.path.exists(output_file):
        os.remove(output_file)

    # 2. Función auxiliar interna para procesar un grupo específico
    def procesar_grupo(grupo_tareas, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_tareas), b_size), desc=descripcion):
            batch = grupo_tareas[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch, system_prompt, modo_salida)
            textos_generados = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                prediccion, explicacion, errors = process_response(texto_bruto, modo_salida)

                tarea_actual = batch[j]
                real = tarea_actual["real"]
                nivel = tarea_actual["nivel"]
                es_correcto = (prediccion == real)

                batch_results.append({
                    "questionId": tarea_actual["id"],
                    "nivel": nivel,
                    "pregunta": tarea_actual["pregunta"],
                    "respuesta_real": real,
                    "prediccion_modelo": prediccion,
                    "explicacion": explicacion,
                    "error_procesamiento_json": errors,
                    "estado": "CORRECTO" if es_correcto else "INCORRECTO"
                })

            # Guardar directamente al fichero
            with open(output_file, 'a', encoding='utf-8') as f:
                for resultado in batch_results:
                    linea_json = json.dumps(resultado, ensure_ascii=False)
                    f.write(linea_json + '\n')

    # 3. Procesar primero los textos (Usando el batch_size configurado)
    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} tareas de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    # 4. Procesar luego las imágenes (Forzando Batch Size = 1)
    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} tareas MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

In [ ]:
def calculate_metrics(input_file="resultados.jsonl"):
    """Lee las predicciones almacenadas y calcula las métricas finales."""

    if not os.path.exists(input_file):
        print(f"Error: No se ha encontrado el archivo {input_file}.")
        return

    stats = {"total": 0, "aciertos": 0, "errores_formato": 0, "por_nivel": {}}

    with open(input_file, 'r', encoding='utf-8') as f:
        for linea in f:
            if not linea.strip():
                continue

            # Cargar el JSON de la línea actual
            resultado = json.loads(linea)

            nivel = resultado["nivel"]
            es_correcto = (resultado["estado"] == "CORRECTO")
            error_json = resultado.get("error_procesamiento_json", False)

            # Inicializar el nivel si no existe en las estadísticas
            if nivel not in stats["por_nivel"]:
                stats["por_nivel"][nivel] = {"aciertos": 0, "total": 0}

            # Contabilizar globales y por nivel
            stats["total"] += 1
            stats["por_nivel"][nivel]["total"] += 1

            if es_correcto:
                stats["aciertos"] += 1
                stats["por_nivel"][nivel]["aciertos"] += 1

            if error_json:
                stats["errores_formato"] += 1

    # Llamar a tu función original para mostrar los resultados en pantalla/guardarlos
    show_results(stats, input_file)

In [ ]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results')

In [ ]:
gemma4_results_path = os.path.join(gemma4_path, "gemma4_zero_shot_fine_tuning.json")

run_inference(
    model=model,
    tokenizer=tokenizer,
    system_prompt=SYSTEM_PROMPT_BRIEF_REASONING,
    modo_salida="json",
    batch_size_texto=16,
    output_file=gemma4_results_path
)

calculate_metrics(input_file=gemma4_results_path)


--- Procesando 128 tareas de SOLO TEXTO (Batch Size: 16) ---


Progreso Texto:  50%|█████     | 4/8 [02:32<02:06, 31.69s/it]A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
